# Trading App v2: Anchored WFO HITS vs DP by Feature-Family Strategy

Every feature family is trained and traded independently. The first OOS year is 2021, trained on all available data before 2020; each later fold expands the training window through the most recently completed year and tests the next calendar year through 2025.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json, os, sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'app').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
WORKSPACE_ROOT = REPO_ROOT.parent
for repo in (WORKSPACE_ROOT / 'quant-warehouse', WORKSPACE_ROOT / 'quant-orchestrator'):
    if repo.exists() and str(repo) not in sys.path: sys.path.insert(0, str(repo))

from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.research_tools import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import RapidsRandomForestClassifier
from quant_orchestrator.research_tools import build_strategy_score_frame, prepare_family_dataset, build_family_prediction_frame
from quant_orchestrator.platforms.backtesting_frameworks.shared_book import SharedBookCostModel, run_shared_book_framework_comparison

WFO_FIRST_TEST_YEAR, WFO_LAST_TEST_YEAR = 2021, 2025
TEST_START, TEST_END = pd.Timestamp(f'{WFO_FIRST_TEST_YEAR}-01-01'), pd.Timestamp(f'{WFO_LAST_TEST_YEAR}-12-31')
TIER_CONFIGS = {'1T': 1_000_000_000_000, '100B': 100_000_000_000, '10B': 10_000_000_000}
_requested_tiers = tuple(x.strip().upper() for x in os.getenv('GRAPH_ORACLE_TIERS', '').split(',') if x.strip())
if _requested_tiers: TIER_CONFIGS = {k: v for k, v in TIER_CONFIGS.items() if k in _requested_tiers}
DP_K = 3
MIN_RETURN = 0.01
HITS_QUANTILE = 0.80
RF_ESTIMATORS = int(os.getenv('GRAPH_ORACLE_FAMILY_RF_ESTIMATORS', '100'))
BACKTEST_VARIANT = os.getenv('GRAPH_ORACLE_VARIANT', 'long_short')
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'graph_oracle_feature_family_wfo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE = {'1T': 'equity_meta_model_1t', '100B': 'equity_meta_model_100b', '10B': 'equity_meta_model_10b'}

def feature_cache_dir(tier):
    cap = TIER_CONFIGS[tier]
    return REPO_ROOT / 'artifacts' / 'trading_app_v2' / BASE_CACHE[tier] / f'mcap_{cap}_train_2020-12-31_seed_20260707' / 'feature_family_panels'


## Build the two target panels

HITS and DP labels are constructed separately within each year from 2021 through 2025. Each annual fold trains on all prior years and tests the next year. DP uses the existing `YE k=3` solver. Feature-family models are trained independently with the canonical RAPIDS random-forest adapter.

In [ ]:
def normalize_prices(raw):
    frame = raw.copy().reset_index() if 'date' not in raw.columns else raw.copy()
    frame.columns = [str(c).lower() for c in frame.columns]
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
    return frame[['date','open','high','low','close']].dropna().sort_values('date').drop_duplicates('date').reset_index(drop=True)

def hits_panel(frame):
    n = len(frame); dates = frame['date'].to_numpy()
    high = pd.to_numeric(frame['high'], errors='coerce').to_numpy(float)
    low = pd.to_numeric(frame['low'], errors='coerce').to_numpy(float)
    valid = np.triu(np.ones((n,n), dtype=bool), 1)
    horizon = np.arange(n)[None,:] - np.arange(n)[:,None]
    valid &= horizon <= 120
    output = {'date': dates}
    for side, returns in [('long', low[None,:] / high[:,None] - 1), ('short', low[:,None] / high[None,:] - 1)]:
        weights = np.where(valid, np.maximum(returns - MIN_RETURN, 0), 0)
        hub = np.ones(n); authority = np.ones(n)
        for _ in range(40):
            authority = authority / (np.linalg.norm(authority) or 1) if not np.allclose(weights.T @ hub, 0) else authority
            authority = weights.T @ hub; authority = authority / (np.linalg.norm(authority) or 1)
            hub = weights @ authority; hub = hub / (np.linalg.norm(hub) or 1)
        hub_cut = np.quantile(hub[hub > 0], HITS_QUANTILE) if (hub > 0).any() else np.inf
        auth_cut = np.quantile(authority[authority > 0], HITS_QUANTILE) if (authority > 0).any() else np.inf
        output[f'hits_{side}_hub'] = (hub >= hub_cut).astype('uint8')
        output[f'hits_{side}_authority'] = (authority >= auth_cut).astype('uint8')
    return pd.DataFrame(output)

def dp_panel(frame):
    out = pd.DataFrame({'date': frame['date']})
    for side in ('long','short'):
        out[f'dp_{side}_entry'] = 0; out[f'dp_{side}_exit'] = 0
    spec = LabelBuildSpec(k_params={'YE':[DP_K]}, min_profit_pct=MIN_RETURN, buy_execution='high', sell_execution='low', short_execution='low', cover_execution='high')
    result = build_trade_results(['S'], spec=spec, price_frames={'S': frame})
    trades = pd.DataFrame(result.completed_trades)
    for side in ('long','short'):
        part = trades.loc[trades['side'].astype(str).str.lower().eq(side)] if not trades.empty else pd.DataFrame()
        entries = set(pd.to_datetime(part['entry_date'], errors='coerce')) if not part.empty else set()
        exits = set(pd.to_datetime(part['exit_date'], errors='coerce')) if not part.empty else set()
        out.loc[out.date.isin(entries), f'dp_{side}_entry'] = 1
        out.loc[out.date.isin(exits), f'dp_{side}_exit'] = 1
    return out

def build_labels(price_frames):
    hits_rows, dp_rows = [], []
    for symbol, frame in price_frames.items():
        for year, part in frame.groupby(frame.date.dt.year):
            if year > WFO_LAST_TEST_YEAR: continue
            h, d = hits_panel(part), dp_panel(part)
            joined = h.merge(d, on='date', how='left'); joined.insert(0, 'symbol', symbol); joined['year'] = year
            hits_rows.append(joined)
    labels = pd.concat(hits_rows, ignore_index=True)
    def side_rows(prefix):
        rows=[]
        for _, r in labels.iterrows():
            for side in ('long','short'):
                positive = (bool(r[f'hits_{side}_hub']) or bool(r[f'hits_{side}_authority'])) if prefix == 'hits' else bool(r[f'dp_{side}_entry'])
                if positive:
                    rows.append({'symbol':r.symbol,'date':r.date,'collapsed_label':f'oracle_{side}'})
        out=pd.DataFrame(rows).drop_duplicates()
        if out.empty: return out
        counts=out.groupby(['symbol','date']).collapsed_label.nunique()
        conflicts=counts[counts.gt(1)].index
        return out.set_index(['symbol','date']).drop(index=conflicts, errors='ignore').reset_index()
    return labels, side_rows('hits'), side_rows('dp')


## Train one strategy per feature family and backtest

Each family is a separate model and a separate strategy source. The shared-book backtest uses the repository's standard daily next-return accounting, `long_short`, `top_k=20`, and 5.5 bps total cost.

In [ ]:
def run_family_tier(tier, cap):
    started=perf_counter(); warehouse=Warehouse()
    cache=feature_cache_dir(tier); index=pd.read_csv(cache/'index.csv')
    first=pd.read_parquet(index.iloc[0].panel_path); symbols=sorted(first.symbol.astype(str).str.upper().unique())
    price_frames={}
    for symbol in symbols:
        raw=warehouse.read_prices(symbol, provider='fmp', start='2024-01-01', end='2025-12-31')
        if raw is not None and not raw.empty:
            try: price_frames[symbol]=normalize_prices(raw)
            except Exception: pass
    labels, hits_targets, dp_targets=build_labels(price_frames)
    print({'tier':tier,'label_rows':len(labels),'hits_target_rows':len(hits_targets),'dp_target_rows':len(dp_targets),'hits_class_counts':hits_targets.collapsed_label.value_counts().to_dict() if not hits_targets.empty else {},'dp_class_counts':dp_targets.collapsed_label.value_counts().to_dict() if not dp_targets.empty else {}})
    results=[]
    wide_close=pd.DataFrame({s: f.set_index('date')['close'] for s,f in price_frames.items()}).sort_index().ffill()
    next_returns=wide_close.pct_change().shift(-1)
    oos_dates=pd.DatetimeIndex(next_returns.index[(next_returns.index>=TEST_START)&(next_returns.index<=TEST_END)])
    for _, meta in index.iterrows():
        family_path=Path(meta.panel_path)
        if not family_path.exists(): continue
        panel=pd.read_parquet(family_path)
        metadata=pd.read_parquet(meta.metadata_path)
        source, family=str(meta.source), str(meta.family)
        family_features=[c for c in metadata.feature.astype(str) if c in panel.columns]
        if not family_features: continue
        for label_name, target_rows in [('hits', hits_targets), ('dp', dp_targets)]:
            train_frame, usable=prepare_family_dataset(panel, metadata.assign(source=source, family=family), target_rows, source=source, family=family, min_feature_coverage=0.50)
            train_frame=train_frame.loc[pd.to_datetime(train_frame.date).between(TRAIN_START,TRAIN_END)]
            if len(train_frame)<50 or train_frame.collapsed_label.nunique()<2: continue
            med=train_frame[usable].median().replace([np.inf,-np.inf],np.nan).fillna(0)
            train_frame[usable]=train_frame[usable].replace([np.inf,-np.inf],np.nan).fillna(med).astype('float32')
            model=RapidsRandomForestClassifier.fit(train_frame, features=usable, target_col='collapsed_label', random_state=20260715, params={'n_estimators':RF_ESTIMATORS,'max_depth':16,'max_features':'sqrt','n_bins':128,'n_streams':8})
            pred=build_family_prediction_frame(panel, usable, min_feature_coverage=0.50)
            pred=pred.loc[pd.to_datetime(pred.date).between(TEST_START,TEST_END)].copy()
            pred[usable]=pred[usable].replace([np.inf,-np.inf],np.nan).fillna(med).astype('float32')
            proba=model.predict_proba_frame(pred, usable)
            scores=build_strategy_score_frame(source=source, family=family, prediction_frame=pred[['symbol','date']], probability_frame=proba, apply_ae_to_exits=False)
            summary, _, _=run_shared_book_framework_comparison(scores=scores, next_returns=next_returns, symbols=tuple(price_frames), dates=oos_dates, variants=('long_short',), top_k_values=(20,), entry_threshold=0.5, exit_threshold=0.5, cost_models={'family_common':SharedBookCostModel(commission_bps=0.5,slippage_bps=5.0)})
            if not summary.empty:
                row=summary.iloc[0].to_dict(); row.update({'tier':tier,'label_source':label_name,'source':source,'family':family,'strategy_source':f'{source}.{family}','train_rows':len(train_frame),'train_long_rate':float(train_frame.collapsed_label.eq('oracle_long').mean()),'train_short_rate':float(train_frame.collapsed_label.eq('oracle_short').mean())}); results.append(row)
            del train_frame, pred, proba, model
    out=pd.DataFrame(results); out.to_parquet(OUTPUT_DIR/f'{tier.lower()}_feature_family_results.parquet',index=False)
    print({'tier':tier,'symbols':len(price_frames),'families':int(out.family.nunique()) if not out.empty else 0,'rows':len(out),'seconds':round(perf_counter()-started,1)})
    return out

def run_family_tier_anchored(tier, cap):
    started=perf_counter(); warehouse=Warehouse()
    cache=feature_cache_dir(tier); index=pd.read_csv(cache/'index.csv')
    first=pd.read_parquet(index.iloc[0].panel_path); symbols=sorted(first.symbol.astype(str).str.upper().unique())
    price_frames={}
    for symbol in symbols:
        raw=warehouse.read_prices(symbol, provider='fmp', start='1900-01-01', end=f'{WFO_LAST_TEST_YEAR}-12-31')
        if raw is not None and not raw.empty:
            try: price_frames[symbol]=normalize_prices(raw)
            except Exception: pass
    labels, hits_targets, dp_targets=build_labels(price_frames)
    print({'tier':tier,'label_rows':len(labels),'hits_target_rows':len(hits_targets),'dp_target_rows':len(dp_targets)})
    wide_close=pd.DataFrame({s: f.set_index('date')['close'] for s,f in price_frames.items()}).sort_index().ffill()
    next_returns=wide_close.pct_change().shift(-1)
    oos_dates=pd.DatetimeIndex(next_returns.index[(next_returns.index>=TEST_START)&(next_returns.index<=TEST_END)])
    results=[]
    for _, meta in index.iterrows():
        family_path=Path(meta.panel_path)
        if not family_path.exists(): continue
        panel=pd.read_parquet(family_path)
        metadata=pd.read_parquet(meta.metadata_path)
        source, family=str(meta.source), str(meta.family)
        family_features=[c for c in metadata.feature.astype(str) if c in panel.columns]
        if not family_features: continue
        for label_name, target_rows in [('hits', hits_targets), ('dp', dp_targets)]:
            fold_scores=[]; fold_train_rows=[]; fold_count=0
            for test_year in range(WFO_FIRST_TEST_YEAR, WFO_LAST_TEST_YEAR + 1):
                train_end=pd.Timestamp('2020-01-01') if test_year == WFO_FIRST_TEST_YEAR else pd.Timestamp(f'{test_year}-01-01')
                test_end=pd.Timestamp(f'{test_year}-12-31')
                train_frame, usable=prepare_family_dataset(panel, metadata.assign(source=source, family=family), target_rows, source=source, family=family, min_feature_coverage=0.50)
                train_frame=train_frame.loc[pd.to_datetime(train_frame.date) < train_end]
                if len(train_frame)<50 or train_frame.collapsed_label.nunique()<2: continue
                med=train_frame[usable].median().replace([np.inf,-np.inf],np.nan).fillna(0)
                train_frame[usable]=train_frame[usable].replace([np.inf,-np.inf],np.nan).fillna(med).astype('float32')
                model=RapidsRandomForestClassifier.fit(train_frame, features=usable, target_col='collapsed_label', random_state=20260715 + test_year, params={'n_estimators':RF_ESTIMATORS,'max_depth':16,'max_features':'sqrt','n_bins':128,'n_streams':8})
                pred=build_family_prediction_frame(panel, usable, min_feature_coverage=0.50)
                pred=pred.loc[pd.to_datetime(pred.date).between(pd.Timestamp(f'{test_year}-01-01'),test_end)].copy()
                pred[usable]=pred[usable].replace([np.inf,-np.inf],np.nan).fillna(med).astype('float32')
                proba=model.predict_proba_frame(pred, usable)
                fold_scores.append(build_strategy_score_frame(source=source, family=family, prediction_frame=pred[['symbol','date']], probability_frame=proba, apply_ae_to_exits=False))
                fold_train_rows.append(len(train_frame)); fold_count += 1
                del train_frame, pred, proba, model
            if not fold_scores: continue
            scores=pd.concat(fold_scores, ignore_index=True)
            summary, _, _=run_shared_book_framework_comparison(scores=scores, next_returns=next_returns, symbols=tuple(price_frames), dates=oos_dates, variants=(BACKTEST_VARIANT,), top_k_values=(20,), entry_threshold=0.5, exit_threshold=0.5, cost_models={'family_common':SharedBookCostModel(commission_bps=0.5,slippage_bps=5.0)})
            if not summary.empty:
                row=summary.iloc[0].to_dict(); row.update({'tier':tier,'label_source':label_name,'source':source,'family':family,'strategy_source':f'{source}.{family}','folds':fold_count,'mean_train_rows':float(np.mean(fold_train_rows)),'last_train_rows':int(fold_train_rows[-1])}); results.append(row)
    out=pd.DataFrame(results); out.to_parquet(OUTPUT_DIR/f'{tier.lower()}_anchored_{BACKTEST_VARIANT}_feature_family_results.parquet',index=False)
    print({'tier':tier,'symbols':len(price_frames),'families':int(out.family.nunique()) if not out.empty else 0,'rows':len(out),'seconds':round(perf_counter()-started,1)})
    return out

all_results=[]
for tier, cap in TIER_CONFIGS.items():
    all_results.append(run_family_tier_anchored(tier,cap))
results=pd.concat(all_results,ignore_index=True)
results.to_csv(OUTPUT_DIR/f'all_anchored_{BACKTEST_VARIANT}_feature_family_results.csv',index=False)
display(results.sort_values(['tier','label_source','sharpe'],ascending=[True,True,False]).head(100))


## Analysis

The family-level comparison is made within the same tier and label source. HITS wins only if it improves OOS Sharpe/return for the same family and the tier-level aggregate is better across the same feature-family strategy set.

In [ ]:
summary=(results.groupby(['tier','label_source'],as_index=False).agg(families=('family','nunique'),mean_return=('total_return','mean'),median_return=('total_return','median'),mean_sharpe=('sharpe','mean'),median_sharpe=('sharpe','median'),mean_drawdown=('max_drawdown','mean'),mean_trades=('trades','mean')).sort_values(['tier','label_source']))
display(summary)
pivot=summary.pivot(index='tier',columns='label_source',values=['mean_return','median_return','mean_sharpe','median_sharpe'])
display(pivot)
